# Applying TF-IDF to our Dataset

In this context, one document will be one issue, along with all it's comments. One corpus will thus be one repository.

### Imports

In [ ]:
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import re
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
import matplotlib.pyplot as plt

### Loading the data into a pandas dataframe

In [ ]:
def load_data(data_path: str) -> pd.DataFrame:
    """
    Loads issue and comment data from a text file into a pandas DataFrame.
    Each issue starts with a line "issue:<opener>", followed by the title and body.
    Each comment starts with a line "comment:<commenter>", followed by the body.
    Args:
        data_path (str): Path to the data file.
    Returns:
        pd.DataFrame: DataFrame containing the issues and comments.
    """
    rows = []

    f =  open(data_path, "r", encoding="utf-8")
    lines = f.read().splitlines()

    f.close()

    i = 0
    issue_id = 0
    n = len(lines)
    current_issue_opener = None

    while i < n:
        line = lines[i]

        if line.startswith("issue:"):
            issue_id += 1

            opener = line[len("issue:"):]
            issue_type = lines[i + 1]
            date = lines[i + 2]
            title = lines[i + 3]
            body = lines[i + 4]

            current_issue_opener = opener

            rows.append({
                "type": "issue",
                "issue_type": (issue_type if issue_type else "N/A"),
                "issue_date": date,
                "opener": opener,
                "title": title,
                "body": body,
                # Issues are opened by the opener, put it in commenter column for easy comparison.
                "commenter": opener,
                "issue_id": issue_id
            })

            i += 5
            continue

        if line.startswith("comment:"):
            commenter = line[len("comment:"):]
            body = lines[i + 1]

            rows.append({
                "type": "comment",
                "opener": current_issue_opener,
                "title": None,
                "body": body,
                "commenter": commenter,
                "issue_id": issue_id
            })

            i += 2
            continue

        i += 1
    df = pd.DataFrame(rows)
    return df

In [ ]:
data_paths = ["raw_data_networkx.txt", "raw_data_pandas.txt", "raw_data_tensorflow.txt"]

dataframes = [load_data(path) for path in data_paths]

# Individual dataframes
df_nx = dataframes[0]
df_pd = dataframes[1]
df_tf = dataframes[2]

### Start applying TF-IDF

In [ ]:
# Each issue is one document, combinding title and body + comments

def combine_issue_comments(df):
    text_per_issue = []

    # We use pandas groupby for faster processing
    grouped_per_issue = df.groupby("issue_id")
    for issue_id, group in grouped_per_issue:
        # Take the text from the issue
        issue_row = group[group["type"] == "issue"].iloc[0]
        issue_text = issue_row["title"] + "\n" + issue_row["body"]

        # Take the text from the comments
        comments = group[group["type"] == "comment"]
        comments_text = "\n".join(comments["body"].tolist())

        # Combine issue text and comments text
        full_text = issue_text + " " + comments_text

        # Append
        text_per_issue.append((issue_id, full_text))
    return text_per_issue

In [ ]:
nx_text_per_issue = combine_issue_comments(df_nx)
pd_text_per_issue = combine_issue_comments(df_pd)
tf_text_per_issue = combine_issue_comments(df_tf)

print(f"""Number of issues in 
      NetworkX dataset: {len(nx_text_per_issue)}
      Pandas dataset: {len(pd_text_per_issue)}
      TensorFlow dataset: {len(tf_text_per_issue)}""")
print(nx_text_per_issue[0])

#### Text Pre-Processing

For text pre-processing, we will apply three parts: Regex filtering, Stopword filtering and Stemming. 

In [ ]:
lemmatizer = WordNetLemmatizer()

regex_textfilter = re.compile(r"[^a-zA-Z\s]")

github_and_general_stopwords = {
    "actually","basically","usually","probably","maybe","somehow","someone",
    "something","anyone","everyone","hi","hello","hey","please","kind",
    "kindof","kinda","sort","sorta","anyway","usecase","usecases","stuff",
    "things","thing","stufflike","thingy","etc","etcetera","btw","currently",
    "previously","literally","essentially","generally","typically","honestly",
    "technically","again","already","still","quite","very","really","just",
    "though","however","meanwhile","from","original","ticket","comment",
    "in","issue", "issues", "the","a","an","and","or","but","if","then","else","when","while","for",
    "of","in","on","at","to","from","by","with","about","into","over","after",
    "before","between","through","during","above","below","up","down","out",
    "off","under","again","further","once","here","there","where","why","how",
    "all","any","both","each","few","more","most","other","some","such","no",
    "nor","not","only","own","same","so","than","too","very","can","will",
    "just","is","am","are","was","were","be","being","been","do","does","did",
    "has","have","had","it","its","this","that","these","those","he","him",
    "his","she","her","they","them","their","we","us","our","you","your","i",
    "me","my"
}


def light_stem(word: str) -> str:
    if word.endswith("ing") and len(word) > 4:
        return word[:-3]
    if word.endswith("ed") and len(word) > 3:
        return word[:-1]
    if word.endswith("es") and len(word) > 3:
        return word[:-1]
    return word

def preprocess_text(text: str) -> str:
    # Handle NaN / None / non-string values
    if not isinstance(text, str):
        return ""

    text = regex_textfilter.sub("", text).lower()
    tokens = text.split()

    cleaned = []
    for w in tokens:
        if w in github_and_general_stopwords:
            continue
        
        w = light_stem(w)
        w = lemmatizer.lemmatize(w)

        # remove single-character tokens
        if len(w) == 1:
            continue

        cleaned.append(w)

    return " ".join(cleaned)


In [ ]:
nx_preprocesed_text = [
    (i, preprocess_text(t))
    for i, t in tqdm(nx_text_per_issue, desc="Processing NetworkX")
]

pd_preprocesed_text = [
    (i, preprocess_text(t))
    for i, t in tqdm(pd_text_per_issue, desc="Processing Pandas")
]

tf_preprocesed_text = [
    (i, preprocess_text(t))
    for i, t in tqdm(tf_text_per_issue, desc="Processing TensorFlow")
]
    
print(f"Issue number {nx_preprocesed_text[0][0]} preprocessed text: {nx_preprocesed_text[0][1]}")

### Save pre-processed text locally

In [ ]:
pd.DataFrame(nx_preprocesed_text, columns=["issue_id", "clean_text"]).to_csv("networkx_preprocessed.csv", index=False)

pd.DataFrame(pd_preprocesed_text, columns=["issue_id", "clean_text"]).to_csv("pandas_preprocessed.csv", index=False)

pd.DataFrame(tf_preprocesed_text, columns=["issue_id", "clean_text"]).to_csv("tensorflow_preprocessed.csv", index=False)

### Calculate document frequencies per token

In [ ]:
# Load preprocessed data from CSV files
nx_preprocesed_text = pd.read_csv("networkx_preprocessed.csv")
pd_preprocesed_text = pd.read_csv("pandas_preprocessed.csv")
tf_preprocesed_text = pd.read_csv("tensorflow_preprocessed.csv")

In [ ]:
def get_vocab(df):
    vocab = set()
    for text in df["clean_text"]:
        if not isinstance(text, str):
            continue
        tokens = text.split()
        vocab.update(tokens)
    return vocab

def get_doc_freqs(preprocessed_data, vocab):
    doc_freqs = {word: 0 for word in vocab}
    for text in preprocessed_data["clean_text"]:
        if not isinstance(text, str):
            continue
        tokens = set(text.split())
        for token in tokens:
            if token in doc_freqs:
                doc_freqs[token] += 1
    return doc_freqs

In [ ]:
nx_vocab = get_vocab(nx_preprocesed_text)
pd_vocab = get_vocab(pd_preprocesed_text)
tf_vocab = get_vocab(tf_preprocesed_text)

nx_doc_freqs = get_doc_freqs(nx_preprocesed_text, nx_vocab)
pd_doc_freqs = get_doc_freqs(pd_preprocesed_text, pd_vocab)
tf_doc_freqs = get_doc_freqs(tf_preprocesed_text, tf_vocab)

print(f"NetworkX vocab size: {len(nx_vocab)}")
print(f"NetworkX first 10 doc freqs: {list(nx_doc_freqs.items())[:-10]}")

### Calculate token frequency per document

In [ ]:
def get_term_freqs_per_issue(preprocessed_data, vocab):
    term_freqs_per_issue = {}
    for _, row in preprocessed_data.iterrows():
        issue_id = row["issue_id"]
        text = row["clean_text"]
        if not isinstance(text, str):
            continue
        tokens = text.split()
        total_terms = len(tokens)
        term_freqs = {}
        for token in tokens:
            if token in vocab:
                term_freqs[token] = term_freqs.get(token, 0) + 1
        for token in term_freqs:
            term_freqs[token] /= total_terms
        term_freqs_per_issue[issue_id] = term_freqs
    return term_freqs_per_issue

nx_df_processed = pd.DataFrame(nx_preprocesed_text, columns=["issue_id", "clean_text"])
pd_df_processed = pd.DataFrame(pd_preprocesed_text, columns=["issue_id", "clean_text"])
tf_df_processed = pd.DataFrame(tf_preprocesed_text, columns=["issue_id", "clean_text"])

nx_term_freqs_per_issue = get_term_freqs_per_issue(nx_df_processed, nx_vocab)
pd_term_freqs_per_issue = get_term_freqs_per_issue(pd_df_processed, pd_vocab)
tf_term_freqs_per_issue = get_term_freqs_per_issue(tf_df_processed, tf_vocab)



### Calculate the TF-IDF

We now have all the information to calculate TF-IDF. N, the amount of documents. TF, the term frequency per document. DF, document of frequency for every term.

In [ ]:
def compute_tfidf(df_processed, vocab, doc_freqs, term_freqs_per_issue):
    N = len(df_processed)
    tfidf_result = {}

    for issue_id, term_freqs in term_freqs_per_issue.items():
        tfidf_scores = {}
        for term, tf in term_freqs.items():
            df = doc_freqs.get(term, 0)
            idf = np.log((N) / (df + 1))
            tfidf_scores[term] = tf * idf
        tfidf_result[issue_id] = tfidf_scores

    return tfidf_result

In [ ]:
nx_tfidf = compute_tfidf(nx_df_processed, nx_vocab, nx_doc_freqs, nx_term_freqs_per_issue)
pd_tfidf = compute_tfidf(pd_df_processed, pd_vocab, pd_doc_freqs, pd_term_freqs_per_issue)
tf_tfidf = compute_tfidf(tf_df_processed, tf_vocab, tf_doc_freqs, tf_term_freqs_per_issue)

In [ ]:
print(f"TF-IDF for issue 1 in NetworkX dataset: {tf_idf[1]}")

## TF-IDF for all repo's together